# Backtest Donchian Channel Breakout

Run the Donchian Channel Breakout (Turtle-style) strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Strategy logic:**
- **Entry:** Close breaks above entry channel upper (new N-period high) → Long. Close breaks below entry channel lower (new N-period low) → Short.
- **Exit:** Close drops below exit channel lower → close long. Close rises above exit channel upper → close short.
- Entry channel (longer period) captures major breakouts. Exit channel (shorter period) acts as a tighter trailing exit.
- Uses crossover detection — only enters on the breakout bar, not on every bar above the channel.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + dual DC bands + trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — entry period vs exit period grid

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.indicators import DonchianChannel
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.donchian_breakout import DonchianBreakout, DonchianBreakoutConfig

from charts import (
    plot_donchian_breakout, plot_equity_curve, print_summary_stats,
    plot_pnl_heatmap, generate_backtest_html,
)
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "4h"
STARTING_CAPITAL = 10_000
TRADE_NOTIONAL   = Decimal("1000")   # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = False

# DC params
ENTRY_PERIOD = 30
EXIT_PERIOD  = 15

# Sweep grids
ENTRY_PERIODS = [10, 15, 20, 25, 30, 40, 50, 55, 60, 65, 70, 75]
EXIT_PERIODS  = [5, 7, 10, 12, 15, 20, 25]

# Standard
INSTRUMENT_ID = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR  = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR  = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE         = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"DonchianBreakout_{INSTRUMENT_ID}_{BAR_INTERVAL}_e{ENTRY_PERIOD}_x{EXIT_PERIOD}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add DonchianBreakout strategy + run ──────────────────────────
config = DonchianBreakoutConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    entry_period=ENTRY_PERIOD,
    exit_period=EXIT_PERIOD,
)
strategy = DonchianBreakout(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"DC(entry={ENTRY_PERIOD}, exit={EXIT_PERIOD}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Plotly chart ───────────────────────────────────────────────────

fig = plot_donchian_breakout(
    bars, fills_report,
    entry_period=ENTRY_PERIOD,
    exit_period=EXIT_PERIOD,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"DC(entry={ENTRY_PERIOD}, exit={EXIT_PERIOD})  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ───────────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep — entry period vs exit period ──────────────
#
# Grid search over entry/exit channel period combinations.
# Constraint: exit_period < entry_period.

def donchian_factory(eng, params):
    cfg = DonchianBreakoutConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        entry_period=params["entry_period"],
        exit_period=params["exit_period"],
    )
    eng.add_strategy(DonchianBreakout(cfg))

combos = [
    {"entry_period": e, "exit_period": x}
    for e in ENTRY_PERIODS
    for x in EXIT_PERIODS
    if x < e
]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=donchian_factory,
    strategy_name="DonchianBreakout",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap (exit period vs entry period) ─────────────────
plot_pnl_heatmap(
    results_df, row_col="exit_period", col_col="entry_period",
    row_label="Exit Period", col_label="Entry Period",
    title=f"Total PnL ({CURRENCY}) by Donchian Channel Parameters",
)

In [ ]:
# ── Cell 13: Best params summary ────────────────────────────────────────
#
# Only 2 params (entry_period, exit_period) so a single sweep covers
# the full grid — no second-stage sweep needed.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
print(f"Best params: entry_period={int(best_row['entry_period'])}, "
      f"exit_period={int(best_row['exit_period'])}")
print(f"  PnL: {best_row['total_pnl']:,.2f}")
print(f"  Positions: {int(best_row['num_positions'])}")
print(f"  Final balance: {best_row['final_balance']:,.2f}")

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ──────────────────────────

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=ENTRY_PERIOD,
    slow_period=EXIT_PERIOD,
    overlay_type="donchian",
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Save notebook snapshot ──────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_donchian_breakout.ipynb", RESULT_NAME)
#save_notebook_html("backtest_donchian_breakout.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 16: Cleanup ──────────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")